In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
import fsspec

# ── Load Data ──────────────────────────────────────────────────────────────────
def load_specific_parquet(url):
    with fsspec.open(url, 'rb') as f:
        return pd.read_parquet(f)

url  = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro.parquet"
metro_24_25 = load_specific_parquet(url)

#---------------Loading the corridor list and cleaning it -------------------------
flight_route = pd.read_excel("Flight Paris for Size the Prize v1.xlsx")
corridors = flight_route[~flight_route['Row Labels'].str.contains('Unknown', na=False)]['Row Labels'].str.strip().iloc[:-1]
corridors = corridors.str.replace(' - ','->')

# Filter out corridors where From and To are identical
corridors = corridors[corridors.apply(lambda x: x.split('->')[0] != x.split('->')[1])]

# copying the Data
Data = metro_24_25.copy()


In [11]:
import pandas as pd
import numpy as np

# ── Precompute lookup arrays outside the loop ──────────────────────────────────────────
Thresh = False
JET_TYPE = 'Super Midsize Jet'
DAY_ORDER    = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
BUCKET_ORDER = ['07:00–10:00', '10:00–13:00', '13:00–16:00',
                '16:00–19:00', '19:00–24:00', '00:00–07:00']

# Vectorized hour → bucket via numpy array indexing (hours 0–23)
_HOUR_TO_BUCKET = np.array(
    ['00:00–07:00'] * 7 +   # hours 0–6
    ['07:00–10:00'] * 3 +   # hours 7–9
    ['10:00–13:00'] * 3 +   # hours 10–12
    ['13:00–16:00'] * 3 +   # hours 13–15
    ['16:00–19:00'] * 3 +   # hours 16–18
    ['19:00–24:00'] * 5     # hours 19–23
)

# Vectorized pct → tier via pd.cut (replaces map + lambda)
TIER_BINS   = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, float('inf')]
TIER_LABELS = ['0 - 1','1 - 2','2 - 3','3 - 4','4 - 5',
               '5 - 6','6 - 7','7 - 8','8 - 9','9 - 10','10+']

# ── Preprocess ONCE before the loop ───────────────────────────────────────────────────
data = Data.iloc[:, 1:].copy()

# filter all the corridors that have same FromMetro and ToMetro
data = data[data['FromMetro'] != data['ToMetro']]

# filtering the Aircraft type 
data = data[data['aircraft_segment'].isin([JET_TYPE])]

# Build route column
data['Route'] = data['FromMetro'] + '->' + data['ToMetro']

# Parse datetime once
data['dep_dt']   = pd.to_datetime(data['FlightDate_ET'], errors='coerce')
data['dep_hour'] = data['dep_dt'].dt.hour

# Day name as ordered categorical (created once)
data['day_name'] = pd.Categorical(
    data['dep_dt'].dt.day_name(),
    categories=DAY_ORDER,
    ordered=True
)

# Hour bucket via vectorized numpy indexing (created once)
data['hour_bucket'] = pd.Categorical(
    _HOUR_TO_BUCKET[data['dep_hour'].to_numpy()],
    categories=BUCKET_ORDER,
    ordered=True
)
valid_corridors = list(set(corridors).intersection(data['Route'].unique()))

# ── Filter to only required corridors ─────────────────────────────────────────────────
missing = [cor for cor in corridors if cor not in data['Route'].values]
if missing:
    print(f"Note: The following corridors were not found in data and will be skipped: {missing}")

data_filtered = data[data['Route'].isin(corridors)]

# ── Single groupby across ALL corridors (no loop needed) ──────────────────────────────
final_report = (
    data_filtered
    .groupby(['Route', 'day_name', 'hour_bucket'], observed=False)
    .size()
    .reset_index(name='flight_count')
)

# Total flights per Route + Day
final_report['total_flights'] = (
    final_report
    .groupby('Route')['flight_count']
    .transform('sum')
)

# Percentage
final_report['pct_flights'] = (
    (final_report['flight_count'] / final_report['total_flights']) * 100
)

# Vectorized tier assignment via pd.cut
final_report['Tier_gap'] = pd.cut(
    final_report['pct_flights'],
    bins=TIER_BINS,
    labels=TIER_LABELS,
    right=False
)
# 1. Convert Tier_gap to string to allow the new 'insufficient' labe
final_report['Tier_gap'] = final_report['Tier_gap'].astype(str)

# 2. Update Tier_gap where flight_count is less than 30
if Thresh:
    final_report.loc[final_report['flight_count'] < Thresh, 'Tier_gap'] = 'insufficient'


In [60]:
# DI_SMID = final_report['Tier_gap'].value_counts().reset_index()
# DI_Light_jet = final_report['Tier_gap'].value_counts().reset_index()

In [96]:
DI_Light_Jet = final_report

In [12]:
DI_SMID = final_report

In [13]:
# Define the filename
output_file = 'DI_Analysis_LJ_SMID.xlsx'

# Use ExcelWriter to save multiple sheets
with pd.ExcelWriter(output_file) as writer:
    DI_Light_Jet.to_excel(writer, sheet_name='DI Light Jet', index=False)
    DI_SMID.to_excel(writer, sheet_name='DI SMID', index=False)

print(f"Successfully saved to {output_file}")

Successfully saved to DI_Analysis_LJ_SMID.xlsx


In [47]:
# final_report[final_report['Route']=='New York City->South Florida']

In [40]:
# final_report['Tier_gap'] = final_report['Tier_gap'].str.replace('10 - 16','10+')
final_report['gap_count'] = final_report.groupby('Tier_gap',observed=False)['day_name'].transform('count')

In [49]:
# final_report.to_excel("Flight_count.xlsx",index_label=False)
gap_count.to_excel("Flight_gap_count.xlsx",index_label=False)

In [25]:
data = Data.copy()
data['Route'] = data['FromMetro']+'->'+data['ToMetro']
# ══════════════════════════════════════════════════════════════════════════════
# PARAMETERS
# ══════════════════════════════════════════════════════════════════════════════
SELECTED_YEAR     = [2025,2024]
SELECTED_CORRIDOR = ('South Florida', 'New York City')

# Options: False      → raw flight counts
#          'index'    → % of time bucket total (row-wise)
#          'columns'  → % of daily total (column-wise)
#          'all'      → % of all corridor flights (grand total)
NORMALIZE = 'all'

# ══════════════════════════════════════════════════════════════════════════════
A, B     = SELECTED_CORRIDOR
route_ab = f'{A}->{B}'

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Filter corridor & year (both directions combined)
# ══════════════════════════════════════════════════════════════════════════════
df = data[
    (data['year'].isin(SELECTED_YEAR)) &
    (data['Route'].isin([route_ab]))
].copy()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Extract departure hour
# ══════════════════════════════════════════════════════════════════════════════
df['dep_dt']   = pd.to_datetime(df['FlightDate_ET'], errors='coerce')
df['dep_hour'] = df['dep_dt'].dt.hour

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Custom time buckets (07:00 start, overnight bucket wraps to 07:00)
#
#   07:00–10:00  →  hour in [7,  8,  9]
#   10:00–13:00  →  hour in [10, 11, 12]
#   13:00–16:00  →  hour in [13, 14, 15]
#   16:00–19:00  →  hour in [16, 17, 18]
#   19:00–24:00  →  hour in [19, 20, 21, 22, 23]
#   00:00–07:00  →  hour in [0,  1,  2,  3,  4,  5,  6]  (overnight)
# ══════════════════════════════════════════════════════════════════════════════
BUCKET_ORDER = [
    '07:00–10:00',
    '10:00–13:00',
    '13:00–16:00',
    '16:00–19:00',
    '19:00–24:00',
    '00:00–07:00',   # overnight — shown last
]

def assign_bucket(hour):
    if  7 <= hour < 10: return '07:00–10:00'
    elif 10 <= hour < 13: return '10:00–13:00'
    elif 13 <= hour < 16: return '13:00–16:00'
    elif 16 <= hour < 19: return '16:00–19:00'
    elif 19 <= hour < 24: return '19:00–24:00'
    else:                 return '00:00–07:00'   # 0–6

df['hour_bucket'] = df['dep_hour'].apply(assign_bucket)
df['day_name']    = df['dep_dt'].dt.day_name()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — pd.crosstab with margins (totals) + optional normalize
# ══════════════════════════════════════════════════════════════════════════════
ct = pd.crosstab(
    index   = df['hour_bucket'],
    columns = df['day_name'],
    margins = True,              # adds "All" row and column (weekly totals)
    normalize = NORMALIZE
)

# ── Reorder rows: custom bucket order + "All" at the bottom ───────────────────
ct = ct.reindex(index=BUCKET_ORDER + ['All'], fill_value=0)

# ── Reorder columns: Mon→Sun + "All" at the right ─────────────────────────────
ct = ct.reindex(columns=DAY_ORDER + ['All'], fill_value=0)

# ── Scale to % when normalized ─────────────────────────────────────────────────
if NORMALIZE:
    ct = (ct * 100).round(1)

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY
# ══════════════════════════════════════════════════════════════════════════════
MODE_LABEL = {
    False     : ('Raw Flight Counts',                        'margins = total flights'),
    'index'   : ('% of Time Bucket Total  (Row-wise)',       'Each row sums to 100%'),
    'columns' : ('% of Daily Total  (Column-wise)',          'Each column sums to 100%'),
    'all'     : ('% of All Corridor Flights (Grand Total)',  'All cells sum to 100%'),
}
title, note = MODE_LABEL[NORMALIZE]

print("=" * 75)
print(f"  Corridor : {A} ⟷ {B}  (both directions combined)")
print(f"  Year     : {SELECTED_YEAR}")
print(f"  Mode     : {title}")
print(f"  Note     : {note}")
print("=" * 75)
ct

  Corridor : South Florida ⟷ New York City  (both directions combined)
  Year     : [2025, 2024]
  Mode     : % of All Corridor Flights (Grand Total)
  Note     : All cells sum to 100%


day_name,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,All
hour_bucket,,,,,,,,
07:00–10:00,3.4,2.9,2.9,2.7,2.8,2.4,2.9,19.9
10:00–13:00,4.2,4.1,3.9,3.9,4.0,3.5,4.8,28.3
13:00–16:00,4.0,3.2,3.1,3.4,3.3,2.4,4.8,24.0
16:00–19:00,3.0,2.5,2.1,2.5,2.0,1.4,3.7,17.2
19:00–24:00,1.2,1.2,1.1,1.1,0.6,0.6,1.6,7.4
00:00–07:00,0.9,0.6,0.4,0.4,0.4,0.2,0.3,3.2
All,16.6,14.3,13.5,13.9,13.1,10.5,18.1,100.0
